# Patent Landscaping — Snorkel vs MAS (Colab GPU)

**Just refresh → Runtime: GPU (T4) → Run all, top to bottom.** It is restart-safe:
- data prep + Snorkel labeling are fast and always run;
- each SciBERT model is **restored from Google Drive if present, otherwise trained (~30 min) and saved to Drive**;
- then train/val curves + **threshold calibration & ROC/PR experiments**.

So the **first run trains once (~1 h)**; every later session **restores in seconds** — no retraining.

Pipeline split: MAS labeling (10 keys) was done locally and `DataSet/mas/mas_ranked_scores.csv` is committed, so it arrives with the clone.

In [ ]:
# 1) setup — clone (once) + update + install.  Absolute paths avoid nested clones.
import os
%cd /content
REPO = 'https://github.com/PassionChicken-Leesuin/Patent_Landscaping_Final.git'
if not os.path.exists('/content/Patent_Landscaping_Final'):
    !git clone $REPO
%cd /content/Patent_Landscaping_Final
!git pull
!pip -q install snorkel transformers datasets scikit-learn accelerate

In [ ]:
# 2) mount Google Drive (persistent store for trained models)
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/patent_landscaping_outputs'
os.makedirs(DRIVE, exist_ok=True)
os.makedirs('outputs', exist_ok=True)
print('model store on Drive:', DRIVE)

In [ ]:
# 3) data prep (fast, always run) — dedup + negative pool + Snorkel labels
!python -m scripts.build_dataset
!python -m scripts.run_snorkel

## 4) Get each model: local → Drive → train (in that order), and keep Drive in sync
First run trains both (~30 min each) and saves to Drive. Later runs restore instantly. If a model is only local (e.g. trained in a previous session before this notebook), it is backed up to Drive automatically. Set `FORCE=True` to retrain from scratch.

In [ ]:
import shutil
FORCE = False           # set True to retrain even if a saved model exists
EPOCHS, MAX_LEN = 4, 256

def _backup(arm):
    local, dstore = f'outputs/scibert_{arm}', f'{DRIVE}/scibert_{arm}'
    shutil.copytree(local, dstore, dirs_exist_ok=True)
    if os.path.isfile(f'outputs/metrics_{arm}.json'):
        shutil.copy(f'outputs/metrics_{arm}.json', DRIVE)

def ensure_model(arm):
    local, dstore = f'outputs/scibert_{arm}', f'{DRIVE}/scibert_{arm}'
    if not FORCE and os.path.isfile(f'{local}/config.json'):
        print(f'[{arm}] using local model')
        if not os.path.isfile(f'{dstore}/config.json'):
            print(f'[{arm}] backing up local model to Drive ...'); _backup(arm)
        return
    if not FORCE and os.path.isfile(f'{dstore}/config.json'):
        print(f'[{arm}] restoring from Drive ...')
        shutil.copytree(dstore, local, dirs_exist_ok=True)
        if os.path.isfile(f'{DRIVE}/metrics_{arm}.json'):
            shutil.copy(f'{DRIVE}/metrics_{arm}.json', 'outputs/')
        print(f'[{arm}] restored'); return
    print(f'[{arm}] no saved model — training (~30 min) ...')
    get_ipython().system(f'python -m scripts.run_downstream --arm {arm} --epochs {EPOCHS} --max-len {MAX_LEN}')
    print(f'[{arm}] saving to Drive ...'); _backup(arm); print(f'[{arm}] done')

ensure_model('snorkel')
ensure_model('mas')

In [ ]:
# 5) training/validation curves (baseline Fig 3/4 style). Needs history.json (saved with the model).
from src.downstream.plots import plot_history
for arm in ['snorkel', 'mas']:
    try:
        plot_history(arm)
    except FileNotFoundError:
        print(f'[{arm}] no history.json (model saved before curves feature) — skip')

## 6) Threshold calibration + ROC/PR experiments
At threshold 0.5 both models predict almost all-negative (recall ~6–12%, specificity ~99% everywhere) because the out-of-domain negatives are too easy — a threshold problem, not a capability one (AUC ≈ 0.71–0.75). This tunes the decision threshold on the **validation** split (never the gold set) and re-reports on gold, with ROC/PR + score-distribution plots. Loads the models from `outputs/` — no retraining.

In [ ]:
!python -m scripts.calibrate_eval